# Validação de AHD e HD de superfície no ImageCAS

Confira a implementação em máscaras sintéticas com resultados conhecidos e em
um exame ImageCAS. Para o exame, duas implementações calculam as distâncias
sobre as mesmas máscaras. Os histogramas e as cenas 3D mostram a distribuição
e a localização das discrepâncias entre predição e referência.

Configure `IMAGECAS_BASE_PATH` se o dataset estiver fora dos caminhos padrões
do projeto.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.ticker import MaxNLocator

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
        src_dir = str(candidate / "src")
        if src_dir not in sys.path:
            sys.path.insert(0, src_dir)
        break
else:
    raise FileNotFoundError("Abra o notebook a partir de uma pasta do repositório.")

from utils.comparison_utils.segmentation_eda import artery_surface_distance_cases
from utils.processing.gpu_utils import use_gpu
from utils.project.evaluation.surface_distance_validation import (
    compare_surface_distance_samples,
    reference_surface_distances,
)
from utils.project.results.io import results_filename
from utils.project.runtime.notebook_env import (
    configure_notebook_environment,
    load_notebook_pipeline_config,
    resolve_imagecas_base_path,
)
from utils.segmentation.pipeline.arteries import segment_artery_masks_from_vesselness
from utils.segmentation.pipeline.detection import detect_ostia, locate_and_segment_aorta
from utils.segmentation.pipeline.preprocessing import (
    compute_vesselness,
    load_and_preprocess_image,
)
from utils.utils.metrics import dice_score, sample_surface_distances
from utils.utils.roi import mask_voxel_bbox_slices
from utils.visualization.images.volume import (
    visualize_binary_masks_comparison,
    visualize_label_map_3d,
)

REPO_ROOT = configure_notebook_environment(chdir_to_src=False)

## Configuração

O limiar de distância define as faixas das figuras. A tolerância define a
diferença numérica admitida na conferência entre os métodos. Para consultar
apenas o histograma canônico, execute os imports, a configuração e a célula
correspondente.

In [ ]:
# Exemplos do canônico mid_res/val (média de Dice de 60,09% em 270 exames).
# Dice alto — 899 (79,35%), 873 (78,95%), 730 (77,44%).
# Próximo da média — 869 (60,33%), 981 (60,44%), 437 (60,51%).
# Em torno de 30% — 162 (31,27%), 307 (32,19%), 920 (27,10%).
IMG_ID = 90
RESOLUTION = "mid"  # "mid" ou "high"
HIGH_DISTANCE_MM = 5.0
DISTANCE_TOLERANCE_MM = 1e-8
HISTOGRAM_BINS = 40

CANONICAL_RESOLUTION = "mid_res"  # "mid_res" ou "high_res"
CANONICAL_SPLIT = "train"  # "train", "val" ou "test"

CONFIG_FILE = REPO_ROOT / "config" / "pipeline_config.json"
USE_GPU = None  # None respeita a configuração do pipeline.

PREDICTION_COLOR = "#2F80ED"
REFERENCE_COLOR = "#39B54A"

if not np.isfinite(HIGH_DISTANCE_MM) or HIGH_DISTANCE_MM < 0:
    raise ValueError("HIGH_DISTANCE_MM deve ser um valor finito e não negativo.")
if not np.isfinite(DISTANCE_TOLERANCE_MM) or DISTANCE_TOLERANCE_MM < 0:
    raise ValueError("DISTANCE_TOLERANCE_MM deve ser finita e não negativa.")
if not isinstance(HISTOGRAM_BINS, int) or HISTOGRAM_BINS < 1:
    raise ValueError("HISTOGRAM_BINS deve ser um inteiro positivo.")
if CANONICAL_RESOLUTION not in {"mid_res", "high_res"}:
    raise ValueError("CANONICAL_RESOLUTION deve ser 'mid_res' ou 'high_res'.")
if CANONICAL_SPLIT not in {"train", "val", "test"}:
    raise ValueError("CANONICAL_SPLIT deve ser 'train', 'val' ou 'test'.")

## Definição e casos de controle

Cada voxel da superfície tem pelo menos um vizinho de face fora da máscara.
Medimos a distância em milímetros do centro de cada voxel de superfície ao
centro mais próximo na outra superfície. A **HD** é o maior valor nos dois
sentidos. A **AHD** é a média aritmética das duas médias direcionais.

O cálculo avaliado extrai a superfície por erosão e busca vizinhos por
KD-tree. A conferência conta os seis vizinhos de face e usa uma transformada
de distância euclidiana (EDT). Os controles abaixo verificam superfícies
idênticas, espaçamento anisotrópico com números diferentes de pontos em cada
sentido, e uma máscara vazia.

In [ ]:
control_spacing = (5.0, 3.0, 2.0)
cube = np.ones((3, 3, 3), dtype=bool)
control_prediction = np.zeros((2, 5, 1), dtype=bool)
control_reference = np.zeros_like(control_prediction)
control_prediction[0, 0, 0] = control_prediction[1, 4, 0] = True
control_reference[0, 0, 0] = True

# O deslocamento físico é (5, 12, 0) mm. As distâncias são [0, 13] e [0].
for case, pred, ref, expected in (
    ("Cubo idêntico", cube, cube, (0.0, 0.0)),
    ("Deslocamento anisotrópico", control_prediction, control_reference, (13.0, 3.25)),
):
    measured = sample_surface_distances(pred, ref, control_spacing)
    edt = reference_surface_distances(pred, ref, control_spacing)
    assert edt is not None
    audit = compare_surface_distance_samples(measured, edt)
    distances = measured.hausdorff_distances()
    assert distances is not None, f"{case}: HD e AHD deveriam estar definidas."
    np.testing.assert_allclose(distances, expected, rtol=0, atol=DISTANCE_TOLERANCE_MM)
    np.testing.assert_allclose(
        (audit.reference_hd_mm, audit.reference_ahd_mm),
        expected,
        rtol=0,
        atol=DISTANCE_TOLERANCE_MM,
    )
    assert audit.max_directional_error_mm <= DISTANCE_TOLERANCE_MM
    if case == "Cubo idêntico":
        assert len(measured.prediction_points_mm) == 26
    print(f"{case} | HD = {expected[0]:g} mm | AHD = {expected[1]:g} mm | OK")

empty = np.zeros_like(cube)
assert (
    sample_surface_distances(empty, cube, control_spacing).hausdorff_distances() is None
)
assert reference_surface_distances(empty, cube, control_spacing) is None
print("Máscara vazia | HD e AHD indefinidas | OK")

## Predição da artéria

A localização da aorta e dos óstios e a segmentação arterial usam a configuração
do pipeline. A comparação usa a predição após o pós-processamento morfológico
e o rótulo coronariano (`label == 1`) na mesma grade processada.

In [ ]:
config = load_notebook_pipeline_config(CONFIG_FILE, RESOLUTION)
gpu_requested = config.get("USE_GPU", False) if USE_GPU is None else USE_GPU
config["USE_GPU"] = bool(gpu_requested and use_gpu())
print(
    f"Exame {IMG_ID} | Resolução {RESOLUTION} | "
    f"Processamento {'GPU' if config['USE_GPU'] else 'CPU'}"
)

image_data = load_and_preprocess_image(
    str(IMG_ID), str(resolve_imagecas_base_path()), config
)
lcc_image = image_data["lcc_image"]
reference_mask = image_data["label"] == 1
# Usa o espaçamento efetivo na ordem dos eixos da grade, como no canonical.
spacing_mm = image_data["scaled_spacing"]
aorta_stage = locate_and_segment_aorta(
    lcc_image,
    image_data["downscale_factors"],
    spacing_mm,
    config,
    require_non_empty=True,
)
del image_data

In [ ]:
vesselness_ostia = compute_vesselness(
    lcc_image, config["VESSELNESS_AORTA"], use_gpu=config["USE_GPU"]
)
ostium_left, ostium_right = detect_ostia(
    aorta_stage.aorta_mask, vesselness_ostia, spacing_mm, config
)
del aorta_stage, vesselness_ostia

vesselness_artery = compute_vesselness(
    lcc_image, config["VESSELNESS_ARTERY"], use_gpu=config["USE_GPU"]
)
artery = segment_artery_masks_from_vesselness(
    lcc_image, vesselness_artery, ostium_left, ostium_right, config
)
prediction_mask = artery.final_mask > 0
print(f"Método arterial: {artery.method}")
del artery, lcc_image, vesselness_artery

## Conferência no exame selecionado

A tabela reúne o Dice e a conferência de AHD e HD por EDT. Depois da tabela,
veja a maior diferença absoluta entre as
distâncias calculadas para os mesmos voxels nos dois sentidos. Esta análise
usa o protocolo de superfície registrado nas colunas `artery_ahd_mm` e
`artery_hd_mm` do canônico. AvgHD e HD da máscara inteira, assim como HD95,
são métricas distintas.

In [ ]:
samples = sample_surface_distances(prediction_mask, reference_mask, spacing_mm)
hausdorff = samples.hausdorff_distances()
pred_to_ref_mm = samples.prediction_to_reference_mm
ref_to_pred_mm = samples.reference_to_prediction_mm
edt_samples = reference_surface_distances(prediction_mask, reference_mask, spacing_mm)
check = (
    compare_surface_distance_samples(samples, edt_samples)
    if edt_samples is not None
    else None
)

hd_mm, ahd_mm = hausdorff if hausdorff is not None else (None, None)
distance_rows = (
    ("AHD (mm)", ahd_mm, check.reference_ahd_mm if check is not None else None),
    ("HD (mm)", hd_mm, check.reference_hd_mm if check is not None else None),
)
rows = [("Dice", f"{dice_score(prediction_mask, reference_mask):.2%}", "—", "—")]
for metric, calculated, expected in distance_rows:
    rows.append(
        (
            metric,
            f"{calculated:.6f}" if calculated is not None else "Indisponível",
            f"{expected:.6f}" if expected is not None else "Indisponível",
            f"{abs(calculated - expected):.2e}"
            if calculated is not None and expected is not None
            else "—",
        )
    )
display(
    pd.DataFrame(
        rows, columns=["Métrica", "Calculado", "EDT", "Diferença absoluta"]
    ).set_index("Métrica")
)

if check is None:
    print("AHD e HD indisponíveis: pelo menos uma das máscaras está vazia.")
    if hausdorff is not None:
        print(
            "DIVERGÊNCIA: o cálculo avaliado retornou distâncias para uma máscara vazia."
        )
elif hausdorff is None:
    print("DIVERGÊNCIA: o cálculo avaliado não retornou AHD e HD.")
else:
    if np.isfinite(check.max_directional_error_mm):
        print(f"Maior diferença por voxel: {check.max_directional_error_mm:.2e} mm")
    else:
        print("Não foi possível comparar as distâncias voxel a voxel.")
    agrees = check.max_directional_error_mm <= DISTANCE_TOLERANCE_MM and all(
        calculated is not None
        and expected is not None
        and abs(calculated - expected) <= DISTANCE_TOLERANCE_MM
        for _, calculated, expected in distance_rows
    )
    print(
        f"Conferência: {'OK' if agrees else 'DIVERGÊNCIA'} "
        f"(tolerância de {DISTANCE_TOLERANCE_MM:g} mm)"
    )

## Predição e referência em 3D

As malhas K3D usam a convenção do pipeline, com predição em azul e referência
em verde. As duas superfícies podem ser alternadas pelo menu da cena. As
malhas servem para inspeção visual. Os cálculos acima usam os centros dos
voxels de superfície.

In [ ]:
def plot_artery_masks(prediction, reference, spacing, title):
    """Exibe as malhas e preserva as coordenadas físicas após o recorte."""
    print(
        f"{title} | Predição: {np.count_nonzero(prediction):,} voxels | "
        f"Referência: {np.count_nonzero(reference):,} voxels"
    )
    bbox = mask_voxel_bbox_slices(prediction | reference)
    if bbox is None:
        print("Nenhum voxel para exibir.")
        return None

    # Uma borda de fundo fecha a malha mesmo quando o vaso toca a borda da grade.
    pred_crop = np.pad(prediction[bbox], 1)
    ref_crop = np.pad(reference[bbox], 1)
    pred_color = int(PREDICTION_COLOR[1:], 16)
    ref_color = int(REFERENCE_COLOR[1:], 16)
    if pred_crop.any() and ref_crop.any():
        plot = visualize_binary_masks_comparison(
            ref_crop,
            pred_crop,
            spacing=spacing,
            display_plot=False,
            plot_name=title,
            reference_name="Artéria GT",
            predicted_name="Artéria predita",
            reference_color=ref_color,
            predicted_color=pred_color,
            reference_opacity=0.34,
            predicted_opacity=0.44,
        )
    else:
        mask, name, color = (
            (pred_crop, "Artéria predita", pred_color)
            if pred_crop.any()
            else (ref_crop, "Artéria GT", ref_color)
        )
        plot = visualize_label_map_3d(
            mask,
            {1: (name, color)},
            spacing=spacing,
            plot_name=title,
            display_plot=False,
        )
    offset_mm = (np.array([axis.start for axis in bbox]) - 1) * spacing
    for mesh in plot.objects:
        mesh.vertices = (mesh.vertices + offset_mm).astype(np.float32)
    plot.axes = ["X (mm)", "Y (mm)", "Z (mm)"]
    display(plot)
    return plot


def surface_points_mask(points_mm, shape, spacing):
    """Recoloca os centros de superfície selecionados na grade da máscara."""
    indices = np.rint(points_mm / spacing).astype(np.intp)
    mask = np.zeros(shape, dtype=bool)
    mask[tuple(indices.T)] = True
    return mask

In [ ]:
full_plot = plot_artery_masks(
    prediction_mask,
    reference_mask,
    spacing_mm,
    f"Exame {IMG_ID} · Predição e referência",
)

## Distribuição das distâncias no exame

Os painéis usam os mesmos intervalos e limites de eixo para os dois sentidos.
Cada observação corresponde a um voxel de superfície. A linha tracejada marca
o limiar usado nas cenas de alta e baixa distância.

In [ ]:
if pred_to_ref_mm is None or ref_to_pred_mm is None:
    print("Histograma indisponível: pelo menos uma das superfícies está vazia.")
else:
    distance_max = max(pred_to_ref_mm.max(), ref_to_pred_mm.max())
    edges = np.linspace(0, distance_max or 1.0, HISTOGRAM_BINS + 1)
    fig, axes = plt.subplots(
        1, 2, figsize=(14, 5), sharex=True, sharey=True, layout="constrained"
    )
    fig.suptitle(f"Exame {IMG_ID} · Distâncias de superfície")
    for ax, values, title, color in (
        (axes[0], pred_to_ref_mm, "Predição → referência", PREDICTION_COLOR),
        (axes[1], ref_to_pred_mm, "Referência → predição", REFERENCE_COLOR),
    ):
        ax.hist(values, bins=edges, color=color, alpha=0.8, edgecolor="black")
        ax.axvline(
            HIGH_DISTANCE_MM,
            color="#C62828",
            linestyle="--",
            label=f"Limiar de {HIGH_DISTANCE_MM:g} mm",
        )
        ax.set(title=title, xlabel="Distância ao vizinho mais próximo (mm)")
        ax.grid(axis="y", alpha=0.3)
        ax.set_axisbelow(True)
        ax.legend()
    axes[0].set_ylabel("Voxels de superfície")
    plt.show()

## Distribuição de AHD e HD no run canônico

Cada observação representa um exame com AHD e HD de superfície válidas. Os
histogramas contextualizam os valores na coorte selecionada. A conferência do
cálculo é feita acima, sobre as mesmas máscaras.

In [ ]:
canonical_csv = (
    REPO_ROOT
    / "output"
    / "segmentation"
    / "canonical"
    / CANONICAL_RESOLUTION
    / CANONICAL_SPLIT
    / "current"
    / "numeric"
    / results_filename(CANONICAL_SPLIT)
)
print(f"Fonte: {canonical_csv.relative_to(REPO_ROOT)}")

if not canonical_csv.is_file():
    print("Resultados canônicos indisponíveis para a seleção.")
else:
    canonical_results = pd.read_csv(canonical_csv)
    canonical_cases = artery_surface_distance_cases(canonical_results)
    print(
        f"Exames com AHD e HD de superfície válidas: "
        f"{len(canonical_cases)}/{len(canonical_results)}"
    )
    if canonical_cases.empty:
        print("O CSV selecionado não contém pares válidos de AHD e HD de superfície.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
        fig.suptitle(f"ImageCAS · Canonical {CANONICAL_RESOLUTION}/{CANONICAL_SPLIT}")
        for ax, metric, column, color in (
            (axes[0], "AHD", "artery_ahd_mm", "skyblue"),
            (axes[1], "HD", "artery_hd_mm", "lightgreen"),
        ):
            ax.hist(
                canonical_cases[column],
                bins=HISTOGRAM_BINS,
                color=color,
                alpha=0.8,
                edgecolor="black",
            )
            ax.set(title=metric, xlabel="Distância de superfície (mm)", ylabel="Exames")
            ax.grid(axis="y", alpha=0.3)
            ax.set_axisbelow(True)
            ax.yaxis.set_major_locator(MaxNLocator(integer=True))
        plt.show()

## Regiões com distâncias altas e baixas em 3D

As regiões de distância **alta** (`≥ HIGH_DISTANCE_MM`) são mostradas em
cenas separadas por sentido. A máscara de destino completa dá o contexto
anatômico. A cena de distância **baixa** mostra as regiões abaixo do limiar
nas duas superfícies.

### Distâncias altas da predição para a referência

A cena sobrepõe os voxels de superfície da predição com distância alta à
referência completa.

In [ ]:
if pred_to_ref_mm is None:
    print("Distâncias indisponíveis: pelo menos uma máscara está vazia.")
else:
    high_prediction = surface_points_mask(
        samples.prediction_points_mm[pred_to_ref_mm >= HIGH_DISTANCE_MM],
        prediction_mask.shape,
        spacing_mm,
    )
    high_prediction_plot = plot_artery_masks(
        high_prediction,
        reference_mask,
        spacing_mm,
        f"Predição → referência · ≥ {HIGH_DISTANCE_MM:g} mm",
    )

### Distâncias altas da referência para a predição

A cena sobrepõe os voxels de superfície da referência com distância alta à
predição completa.

In [ ]:
if ref_to_pred_mm is None:
    print("Distâncias indisponíveis: pelo menos uma máscara está vazia.")
else:
    high_reference = surface_points_mask(
        samples.reference_points_mm[ref_to_pred_mm >= HIGH_DISTANCE_MM],
        reference_mask.shape,
        spacing_mm,
    )
    high_reference_plot = plot_artery_masks(
        prediction_mask,
        high_reference,
        spacing_mm,
        f"Referência → predição · ≥ {HIGH_DISTANCE_MM:g} mm",
    )

### Distâncias baixas nas duas superfícies

In [ ]:
if pred_to_ref_mm is None or ref_to_pred_mm is None:
    print("Distâncias indisponíveis: pelo menos uma máscara está vazia.")
else:
    low_prediction = surface_points_mask(
        samples.prediction_points_mm[pred_to_ref_mm < HIGH_DISTANCE_MM],
        prediction_mask.shape,
        spacing_mm,
    )
    low_reference = surface_points_mask(
        samples.reference_points_mm[ref_to_pred_mm < HIGH_DISTANCE_MM],
        reference_mask.shape,
        spacing_mm,
    )
    low_plot = plot_artery_masks(
        low_prediction,
        low_reference,
        spacing_mm,
        f"Distâncias baixas · < {HIGH_DISTANCE_MM:g} mm",
    )